## Scikit-Learn Regressziós modell - labor

Autó árának becslése

In [ ]:
#szükséges könyvtárak importálása
import pandas as pd
import seaborn as sns

In [ ]:
# car_sales.csv adathalmazból dataframe készítése car_sales néven
car_sales = pd.read_csv("car_sales.csv",index_col=0)

In [ ]:
# dataframe áttekintése
car_sales.head()

In [ ]:
car_sales.info()

Az `info()` alapján
* Hány hiányzó értéket tartalmazó adat (sor) van?
* Milyen típusúak a tulajdonságok (oszlopok)?
* Hány hiányzó érték van az egyes oszlopokban?

In [ ]:
# Hiányzó értékek számának meghatározása az egyes oszlopokban
car_sales.isna().sum()

In [ ]:
# Oszlopok adattípusának megjelenítése
car_sales.dtypes

In [ ]:
# Az oszlopok nevei
car_sales.columns

## Regressziós modell készítése

In [ ]:
# Tanítóhalmaz és teszthalmaz készítése (test_size=0.3, random_state=101)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(car_sales[['Make', 'Colour', 'Odometer (KM)', 'Doors']], car_sales['Price'], test_size=0.3, random_state=101)

In [ ]:
# RandomForestRegressor importálása
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# RandomForestRegressor példányosítása
rfr = RandomForestRegressor()

In [ ]:
# Tanítás a tanítóhalmazon
rfr.fit(X_train, y_train)

### Hiányzó értékek és kategorikus értékek kezelése

- hiányzó értékek törlése: `dropna` https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html
- hiányzó értékek helyettesítése: `SimpleImputer` https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html
- oszlopok értékeinek traszformálása: `ColumnTransformer` https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html
- oszlopok kategorikus értékeinek traszformálása numerikus értékekké: `OneHotEncoder` https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

In [ ]:
# A Price oszlopban levő hiányzó értékek (NaN) kezelése: sorok törlése
car_sales.dropna(subset=["Price"], inplace=True)

In [ ]:
car_sales.isna().sum()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
#'Doors' hiányzó értékeinek helyettesítése 4-gyel
car_sales['Doors']=SimpleImputer(strategy="constant", fill_value=4).fit_transform(car_sales[['Doors']])

In [ ]:
car_sales.isna().sum()

In [ ]:
#'Make' és 'Colour' hiányzó értékeinek helyettesítése 'missing'-gel
car_sales[['Make','Colour']]=SimpleImputer(strategy="constant", fill_value='missing').fit_transform(car_sales[['Make','Colour']])

In [ ]:
#'Odometer (KM)' hiányzó értékeinek helyettesítése a mediánnal
car_sales['Odometer (KM)']=SimpleImputer(strategy="median").fit_transform(car_sales[['Odometer (KM)']])

In [ ]:
# OneHotEncoder a kategorikus értékek numerikus értékekké alakításához
enc=OneHotEncoder(sparse_output=False, dtype=int)
enc_array=enc.fit_transform(car_sales[['Make','Colour']])

In [ ]:
enc_array

In [ ]:
enc.get_feature_names_out()

In [ ]:
df_encoded = pd.DataFrame(enc_array, columns=enc.get_feature_names_out(), index=car_sales.index)

In [ ]:
df_encoded

In [ ]:
car_sales_encoded = pd.concat([car_sales,df_encoded],axis=1)

In [ ]:
car_sales_encoded.drop(['Make','Colour'],axis=1,inplace=True)

## Regressziós modell készítése

In [ ]:
car_sales_encoded.columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(car_sales_encoded[['Odometer (KM)', 'Doors', 'Price',
    'Make_BMW', 'Make_Honda', 'Make_Nissan', 'Make_Toyota', 'Make_missing',
    'Colour_Black', 'Colour_Blue', 'Colour_Green', 'Colour_Red', 'Colour_White', 'Colour_missing']],
    car_sales_encoded['Price'], test_size=0.3, random_state=101)

In [ ]:
rfr = RandomForestRegressor()

In [ ]:
rfr.fit(X_train, y_train)

In [ ]:
predictions = rfr.predict(X_test)

In [ ]:
test_df = pd.DataFrame(y_test,columns=['Price'])
predict_df = pd.DataFrame(predictions,columns=['Predict'],index=test_df.index)
result_df = pd.concat([test_df,predict_df],axis=1)
result_df.head()

In [ ]:
sns.scatterplot(x=result_df['Price'],y=result_df['Predict']);

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

In [ ]:
mse = mean_squared_error(result_df['Price'],result_df['Predict'])
mse

In [ ]:
mae =  mean_absolute_error(result_df['Price'],result_df['Predict'])
mae

### Másik modell kipróbálása?

![image.png](attachment:image.png)

https://scikit-learn.org/stable/tutorial/machine_learning_map/index.html